# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 


## **1. Input Formats**

### **1.1. NumPy Array Shapes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_01.jpg?v=1783947742" width="250">



>* Rows are samples; columns are features
>* Correct shape lets estimators interpret data

>* Use 2D arrays for single-feature data
>* Shape single samples as one row

>* Targets must align with feature rows
>* Check shapes before fitting estimators



In [ ]:
#@title Python Code - NumPy Array Shapes

# NumPy shapes define sample feature meaning.
# Flat arrays can confuse estimators.
# Reshaping makes learning intent explicit.

import numpy as np

# Create one feature measured for five apartments.
sizes_sqft = np.array([500, 650, 800, 950, 1100])

# Create one target value per apartment.
rent_usd = np.array([1500, 1800, 2100, 2400, 2700])

# Inspect the ambiguous flat feature shape.
flat_shape = sizes_sqft.shape

# Reshape many samples with one feature.
X_one_feature = sizes_sqft.reshape(-1, 1)

# Create one sample with four separate features.
loan_applicant = np.array([[72000, 690, 0.32, 5]])

# Validate sample counts before fitting later.
matching_rows = X_one_feature.shape[0] == rent_usd.shape[0]

# Print compact shape diagnostics only.
print(f"Flat feature shape: {flat_shape}")
print(f"One-feature matrix shape: {X_one_feature.shape}")
print(f"Target vector shape: {rent_usd.shape}")
print(f"Rows match targets: {matching_rows}")
print(f"One applicant matrix shape: {loan_applicant.shape}")
print("Rule: X is always samples by features.")



### **1.2. DataFrame Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_02.jpg?v=1783947750" width="250">



>* Rows are samples; columns are features
>* Use two-dimensional DataFrames, even for one feature

>* Column names make features easier to validate
>* Keep training and prediction columns consistent

>* Preprocess mixed DataFrame contents before fitting
>* Keep features two-dimensional and target separate



In [ ]:
#@title Python Code - DataFrame Inputs

# DataFrames keep feature tables visibly two dimensional.
# Column names help protect feature meaning.
# Targets usually stay one dimensional.

import pandas as pd

# Build a tiny feature table.
features = pd.DataFrame(
    {
        "age_years": [29, 41, 35],
        "income_usd": [52000, 83000, 61000],
        "distance_miles": [3.2, 12.5, 7.8],
    }
)

# Build a separate target series.
target = pd.Series(
    [0, 1, 0],
    name="churned",
)

# Select one feature as DataFrame.
one_feature_table = features[["age_years"]]

# Select one feature as Series.
one_feature_series = features["age_years"]

# Check the expected shapes.
print("Feature table shape:", features.shape)
print("Target series shape:", target.shape)
print("One-column DataFrame shape:", one_feature_table.shape)
print("One-column Series shape:", one_feature_series.shape)

# Inspect column names and dtypes.
print("Feature columns:", list(features.columns))
print("Feature dtypes:", features.dtypes.to_dict())

# Validate the basic data contract.
valid_rows = features.shape[0] == target.shape[0]
valid_table = len(features.shape) == 2
valid_target = len(target.shape) == 1

# Summarize contract checks compactly.
print("Rows match target:", valid_rows)
print("Features are two dimensional:", valid_table)
print("Target is one dimensional:", valid_target)



### **1.3. Sparse Matrix Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_03.jpg?v=1783947753" width="250">



>* Rows are samples; columns are possible features
>* Store only nonzeros to save memory

>* Choose efficient CSR or CSC formats
>* Keep sparse matrices two-dimensional and consistent

>* Avoid accidentally densifying sparse data
>* Keep numeric features and stable columns



In [ ]:
#@title Python Code - Sparse Matrix Inputs

# Sparse matrices store mostly zero feature tables.
# Rows are samples and columns are features.
# This example checks shape and memory safety.

import numpy as np
from scipy import sparse

# Create tiny row, column, and value arrays.
row_index = np.array([0, 0, 1, 2, 3, 3])
col_index = np.array([0, 3, 1, 2, 0, 4])
values = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])

# Build a compressed sparse row feature matrix.
X_sparse = sparse.csr_matrix((values, (row_index, col_index)), shape=(4, 5))
y_target = np.array([0, 1, 0, 1])

# Validate the estimator-style data contract.
assert X_sparse.ndim == 2
assert X_sparse.shape[0] == y_target.shape[0]
assert np.issubdtype(X_sparse.dtype, np.number)

# Compare sparse storage with dense storage.
dense_bytes = X_sparse.toarray().nbytes
sparse_bytes = X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes

# Convert format only when an operation needs it.
X_column_friendly = X_sparse.tocsc()
format_pair = (X_sparse.getformat(), X_column_friendly.getformat())

# Print compact inspection results for learners.
print(f"Sparse shape: {X_sparse.shape}")
print(f"Target shape: {y_target.shape}")
print(f"Nonzero entries: {X_sparse.nnz}")
print(f"Formats: CSR then CSC = {format_pair}")
print(f"Dense bytes versus sparse bytes: {dense_bytes} vs {sparse_bytes}")
print("Contract satisfied: two-dimensional numeric features, matching target rows.")



## **2. Shapes and Dtypes**

### **2.1. Feature Matrix Shape**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_01.jpg?v=1783947762" width="250">



>* Rows are samples; columns are features
>* Keep one-feature inputs two-dimensional

>* Keep single-feature data two-dimensional
>* Avoid swapping samples and features

>* Match rows to targets and features
>* Check shape after every transformation



In [ ]:
#@title Python Code - Feature Matrix Shape

# Feature matrices need two dimensions.
# Rows are samples, columns features.
# Shape checks prevent fitting mistakes.

import numpy as np

# Create one feature measured for five houses.
house_area_sqft = np.array([850, 920, 1100, 1250, 1400])

# Create one target value per house.
prices_k = np.array([210, 230, 275, 310, 360])

# Show the common one-dimensional mistake.
print("1D feature shape:", house_area_sqft.shape)

# Reshape into samples by one feature.
X_one_feature = house_area_sqft.reshape(-1, 1)

# Check that rows match target values.
rows_match = X_one_feature.shape[0] == prices_k.shape[0]

# Show the corrected feature matrix contract.
print("2D feature shape:", X_one_feature.shape)
print("Target shape:", prices_k.shape)
print("Rows match target:", rows_match)

# Create one sample with three metric features.
one_patient = np.array([172, 78, 120]).reshape(1, -1)

# Show one row with several columns.
print("Single sample shape:", one_patient.shape)

# Create a tiny two-feature matrix directly.
X_two_features = np.array([[850, 2], [920, 2], [1100, 3]])

# Confirm the matrix is two-dimensional.
print("Two-feature matrix shape:", X_two_features.shape)
print("Feature matrix ndim:", X_two_features.ndim)



### **2.2. Target Shape Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_02.jpg?v=1783947756" width="250">



>* Targets must align with feature rows
>* Check target length before fitting

>* Most targets should be one-dimensional
>* Flatten single-column targets to avoid ambiguity

>* Allow true multi-output targets when intended
>* Check rows and dimensions before fitting



In [ ]:
#@title Python Code - Target Shape Checks

# Target arrays must match feature rows.
# One column can still be ambiguous.
# Check shapes before fitting estimators.

import numpy as np
import pandas as pd

# Create a tiny feature table.
X = pd.DataFrame({
    "height_cm": [170, 165, 180, 155],
    "weight_kg": [70, 60, 85, 50],
})

# Create several target shape examples.
y_good = pd.Series([1, 0, 1, 0])
y_column = pd.DataFrame({"renewed": [1, 0, 1, 0]})
y_short = pd.Series([1, 0, 1])
y_multi = pd.DataFrame({
    "price_usd": [200, 150, 300, 120],
    "rating": [5, 4, 5, 3],
})

# Define a compact target checker.
def check_target(name, X_data, y_data):
    x_rows = X_data.shape[0]
    y_shape = np.shape(y_data)

    # Compare sample counts first.
    rows_match = len(y_data) == x_rows
    dims = np.ndim(y_data)

    # Describe likely target meaning.
    if rows_match and dims == 1:
        status = "OK: one target per row"
    elif rows_match and y_shape[1] == 1:
        status = "WARN: flatten single column"
    elif rows_match and y_shape[1] > 1:
        status = "OK: intentional multi-output"
    else:
        status = "ERROR: row count mismatch"

    # Print one concise diagnostic line.
    print(f"{name}: X{X_data.shape}, y{y_shape} -> {status}")

# Run checks on common cases.
check_target("good_1d", X, y_good)
check_target("column_2d", X, y_column)
check_target("short_1d", X, y_short)
check_target("multi_2d", X, y_multi)

# Show the safe flattening operation.
y_flat = y_column.to_numpy().ravel()
print(f"flattened column target shape: {y_flat.shape}")



### **2.3. Dtype Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_03.jpg?v=1783947764" width="250">



>* Correct shapes still need valid dtypes
>* Find text or mixed columns before fitting

>* Check feature dtypes and encode categories
>* Match target dtype to modeling task

>* Missing values can change column dtypes
>* Check sparse and float precision choices



In [ ]:
#@title Python Code - Dtype Checks

# Dtype checks prevent confusing estimator errors.
# Shapes alone cannot prove data readiness.
# Inspect features and targets before fitting.

# Import small, standard data tools.
import numpy as np
import pandas as pd

# Build features with intentional dtype problems.
features = pd.DataFrame({
    "age_years": [25, 41, " 36", 52],
    "height_inches": [68.0, 71.5, 65.2, 70.1],
    "city": ["Austin", "Boston", "Austin", "Denver"],
})

# Build a target with inconsistent labels.
target = pd.Series(
    ["yes", "Yes ", "no", "yes"],
    name="bought_plan",
)

# Show compact dtype information only.
print("Feature dtypes:", features.dtypes.to_dict())
print("Target dtype:", str(target.dtype))

# Identify columns that are not numeric.
numeric_flags = features.apply(
    lambda column: pd.api.types.is_numeric_dtype(column)
)
problem_columns = numeric_flags[~numeric_flags].index.tolist()

# Report likely feature contract violations.
print("Non-numeric feature columns:", problem_columns)
print("Feature shape:", features.shape)

# Clean one numeric-looking text column safely.
features["age_years"] = pd.to_numeric(
    features["age_years"], errors="coerce"
)

# Normalize target labels before counting classes.
clean_target = target.astype(str).str.strip().str.lower()
unique_labels = sorted(clean_target.unique().tolist())

# Recheck dtypes after targeted cleaning.
print("Cleaned age dtype:", str(features["age_years"].dtype))
print("Cleaned target labels:", unique_labels)

# Summarize remaining work before fitting.
remaining_objects = features.select_dtypes(include="object").columns.tolist()
print("Still needs encoding:", remaining_objects)



## **3. Inspection Utilities**

### **3.1. Bunch Data Containers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_01.jpg?v=1783947733" width="250">



>* Bunch stores dataset arrays and metadata together
>* Named access helps interpret data before modeling

>* Bunch preserves metadata before model fitting
>* Context helps verify labels and feature alignment

>* Use Bunches to inspect and prototype datasets
>* Keep data and metadata together for clarity



In [ ]:
#@title Python Code - Bunch Data Containers

# Bunch containers keep dataset pieces together.
# Attribute access makes inspection beginner friendly.
# Metadata helps validate modeling inputs early.

from types import SimpleNamespace
import numpy as np
import pandas as pd

# Create tiny feature data with names.
feature_names = ["height_cm", "weight_kg", "age_years"]
target_names = ["low_risk", "high_risk"]
measurements = np.array([[170, 68, 29], [160, 72, 45], [182, 85, 51]])

# Create a one dimensional target.
target = np.array([0, 1, 1])
description = "Tiny health example for Bunch-style inspection."
health_bunch = SimpleNamespace(data=measurements, target=target, feature_names=feature_names, target_names=target_names, DESCR=description)

# Inspect container fields without printing arrays.
print("Container fields:", sorted(vars(health_bunch).keys()))
print("Feature shape:", health_bunch.data.shape)
print("Target shape:", health_bunch.target.shape)

# Validate common estimator data contracts.
rows_match = health_bunch.data.shape[0] == health_bunch.target.shape[0]
columns_match = health_bunch.data.shape[1] == len(health_bunch.feature_names)
target_is_vector = health_bunch.target.ndim == 1

# Convert to pandas for readable inspection.
frame = pd.DataFrame(health_bunch.data, columns=health_bunch.feature_names)
frame["risk_label"] = [health_bunch.target_names[i] for i in health_bunch.target]
print("Rows match target:", rows_match)

# Show metadata and a compact preview.
print("Columns match names:", columns_match)
print("Target is vector:", target_is_vector)
print("Feature names:", health_bunch.feature_names)
print("Target names:", health_bunch.target_names)
print("Preview row:", frame.iloc[0].to_dict())



### **3.2. Cloning Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_02.jpg?v=1783947735" width="250">



>* Separate model settings from learned results
>* Clone clean estimators for fresh fitting

>* Start each split with a fresh estimator
>* Prevent leakage while preserving model settings

>* Separate setup parameters from learned state
>* Clean cloning reveals API design problems



### **3.3. Feature Name Inspection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_03.jpg?v=1783947737" width="250">



>* Inspect feature names from labeled training data
>* Catch missing, renamed, or reordered columns

>* Match model features to intended inputs
>* Inspect pipelines for debugging and governance

>* Catch production feature changes before prediction
>* Use names for inspection and reporting



# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>


In this lecture, you learned to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 

In the next Module (Module 3), we will go over 'Datasets'